In [1]:
import pandas as pd


In [2]:
df_new_data = pd.read_csv('scraped_center_reviews.csv')
df_new_data.head(5)

,text,stars
0,I would like to share my experience with Glitz...,5
1,I’m very satisfied with the accident repair se...,5
2,My first experience at Glitz Park was exceptio...,5
3,Excellent ambiance and high quality service. T...,5
4,I recently handed over my vehicle to Glits Par...,5


Replace rating with sentiments

In [3]:
print("Dropping null values in stars")
df_new_data = df_new_data.dropna(subset=['stars'])
sentiments = []
print("Find sentiments in datasets")
for index, row in df_new_data.iterrows():
    if row['stars'] > 3 :
        sentiments.append("Good")
    elif row['stars'] < 3 :
        sentiments.append("Bad")
    else :
        sentiments.append("Neutral")
print("Assign sentiments to the dataset")
df_new_data['original_sentiments'] = sentiments
print("Completed!")
df_new_data.drop('stars', axis=1, inplace=True)
df_new_data.head(5)

Dropping null values in stars
Find sentiments in datasets
Assign sentiments to the dataset
Completed!


,text,original_sentiments
0,I would like to share my experience with Glitz...,Good
1,I’m very satisfied with the accident repair se...,Good
2,My first experience at Glitz Park was exceptio...,Good
3,Excellent ambiance and high quality service. T...,Good
4,I recently handed over my vehicle to Glits Par...,Good


In [4]:
import string
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
ps = PorterStemmer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
import joblib

# Load the TF-IDF vectorizer
tfidf = joblib.load('tfidf_vectorizer.pkl')

# Load the trained model
model = joblib.load('multinomial_nb_model.pkl')

In [6]:
from nltk.tokenize import word_tokenize
import re

In [7]:
stop_words = set(stopwords.words('english'))
negation_words = {
    'not', 'no', 'never', 'neither', 'nor', 'but', 'cannot',
    'barely', 'hardly', 'scarcely', 'without', 'against'
}
contraction_remnants = {
    'don', 't', 'doesn', 'didn', 'wasn', 'weren', 'haven', 'hasn',
    'hadn', 'won', 'wouldn', 'shann', 'shouldn', 'can', 'couldn',
    'isn', 'aren', 'ain'
}
all_negations_to_keep = negation_words.union(contraction_remnants)
stop_words = stop_words - all_negations_to_keep

def clean_text(text):
    # Handle missing/null values
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # URL removal
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove punctuation and special characters BEFORE tokenization
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    #Remove servic center name
    text = re.sub(r'glitz park', '', text,)
    text = re.sub(r'glitz', '', text,)

    #Remove names
    text = re.sub(r'mr\.?\s*[a-z]+(\s+[a-z]+)?', '', text, flags=re.IGNORECASE)

    # Tokenize
    words = word_tokenize(text)

    # Filter out stopwords, short noise, and then stem
    filtered_and_stemmed_words = []
    for word in words:
        # Strip extra whitespace just in case
        word = word.strip()
        # Filter out stopwords and single-letter junk tokens (like 'u', 'x', etc.)
        if word and (word not in stop_words) and len(word) > 1:
            filtered_and_stemmed_words.append(ps.stem(word))

    return " ".join(filtered_and_stemmed_words)

In [8]:
df_new_data["clean_text"] = df_new_data["text"].apply(clean_text)
df_new_data.head(5)

,text,original_sentiments,clean_text
0,I would like to share my experience with Glitz...,Good,would like share experi give five star rate ex...
1,I’m very satisfied with the accident repair se...,Good,satisfi accid repair servic provid repair work...
2,My first experience at Glitz Park was exceptio...,Good,first experi except took care vehicl realli ap...
3,Excellent ambiance and high quality service. T...,Good,excel ambianc high qualiti servic staff friend...
4,I recently handed over my vehicle to Glits Par...,Good,recent hand vehicl glit park concern took care...


In [9]:
X_new_test = tfidf.transform(df_new_data['clean_text']).toarray()

In [10]:
y_new_pred = model.predict(X_new_test)

In [11]:
df_new_data['Predicted_sentiment'] = y_new_pred
df_new_data.head(5)

,text,original_sentiments,clean_text,Predicted_sentiment
0,I would like to share my experience with Glitz...,Good,would like share experi give five star rate ex...,Good
1,I’m very satisfied with the accident repair se...,Good,satisfi accid repair servic provid repair work...,Good
2,My first experience at Glitz Park was exceptio...,Good,first experi except took care vehicl realli ap...,Good
3,Excellent ambiance and high quality service. T...,Good,excel ambianc high qualiti servic staff friend...,Good
4,I recently handed over my vehicle to Glits Par...,Good,recent hand vehicl glit park concern took care...,Good


In [12]:
Y_new = df_new_data['original_sentiments'].values

In [13]:
from sklearn.metrics import accuracy_score
print(accuracy_score(Y_new, y_new_pred))

1.0


In [14]:
df_new_data.to_csv('vehicle_reviews_finalized.csv', index=False)